Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Feature Importance
This notebook checks permutation feature importance on a synthetic history whose dependencies are known:

- `driver` moves in lasting levels with small noise around each level. `target` relaxes toward a level set by the previous step's `driver`, closing a fifth of the gap each step. The last level change happens three steps before the forecast, so its effect on `target` is still building, and only the driver's history shows it. That gives `driver` something to contribute beyond the target's own history, which is what importance measures.
- `placebo` is a random walk unrelated to both and serves as the yardstick below.

`client.feature_importance(...)` shuffles one feature column across the history rows, which keeps its values but destroys their order in time, forecasts again, and compares that sample forecast with the one from the untouched history. A forecast reads mostly a feature's last few values, and one shuffle can leave similar values there by chance, so each feature is shuffled `n_repeats` times (five by default) and its scores are averaged. Two scores come out per target and horizon:

- `mean`: the absolute shift of the forecast mean, in the target's units.
- `distance`: the centered squared 2-Wasserstein distance between the two sample sets, which pairs the sorted draws of both sets and averages their squared gaps (halved). Both sets are centered first, so it catches a change in spread, skew, or tails that leaves the mean in place. Its unit is the target's unit squared.

The call sends one sample forecast for the baseline plus `n_repeats` per feature, all at the same `n_samples` and `seed`.

In [ ]:
import numpy as np
import pandas as pd

HISTORY_SEED = 11
STEPS = 100
LEVEL_STEPS = 13
DRIVER_LEVELS = [1.5, 2.5, 4.0]
FINAL_LEVEL = 5.0
FINAL_LEVEL_AGE_STEPS = 3
DRIVER_NOISE = 0.05
TARGET_BASE = 150.0
TARGET_GAIN = 200.0
CATCH_UP = 0.2
TARGET_NOISE = 6.0
PLACEBO_START = 100.0
PLACEBO_VOLATILITY = 0.01


def synthetic_history():
    """Return a history in which `driver` moves `target` and `placebo` moves nothing."""
    rng = np.random.default_rng(HISTORY_SEED)
    level_count = -(-STEPS // LEVEL_STEPS)
    levels = np.repeat(rng.choice(DRIVER_LEVELS, size=level_count), LEVEL_STEPS)[:STEPS]
    levels[-FINAL_LEVEL_AGE_STEPS:] = FINAL_LEVEL
    driver = levels * rng.lognormal(0.0, DRIVER_NOISE, STEPS)
    target_noise = rng.normal(0.0, TARGET_NOISE, STEPS)
    target = np.empty(STEPS)
    target[0] = TARGET_BASE + TARGET_GAIN * driver[0]
    for step in range(1, STEPS):
        gap = TARGET_BASE + TARGET_GAIN * driver[step - 1] - target[step - 1]
        target[step] = target[step - 1] + CATCH_UP * gap + target_noise[step]
    placebo = PLACEBO_START * np.exp(
        np.cumsum(rng.normal(0.0, PLACEBO_VOLATILITY, STEPS))
    )
    return pd.DataFrame({"driver": driver, "placebo": placebo, "target": target})


history = synthetic_history()
history.tail(FINAL_LEVEL_AGE_STEPS + 2)

In [ ]:
from jointfm_client import JointFMClient, plan_forecast_columns

FEATURE_COLUMNS = ["driver", "placebo"]
PLACEBO_COLUMN = "placebo"
TARGET_COLUMNS = ["target"]
N_SAMPLES = 1024
N_REPEATS = 5
OUTPUT_HORIZONS = 10
QUERY_TIMES = list(range(STEPS, STEPS + OUTPUT_HORIZONS))
HORIZONS = list(range(1, OUTPUT_HORIZONS + 1))

client = JointFMClient.from_env()
instance_ids = [
    entry.instance_id for entry in client.health_instances(cache=True).instances
]
print(f"JointFM instance(s): {instance_ids}")
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)
importance = client.feature_importance(
    history,
    query_times=QUERY_TIMES,
    horizons=HORIZONS,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    columns=plan.columns,
    n_samples=N_SAMPLES,
    n_repeats=N_REPEATS,
    seed=7,
)


def importance_scores(entries):
    """Return one row per feature, target, and horizon."""
    return pd.DataFrame(
        [
            {
                "feature": entry["feature"],
                "target": target,
                "horizon": horizon,
                "mean": entry["mean"][target][horizon],
                "distance": entry["distance"][target][horizon],
            }
            for entry in entries
            for target in TARGET_COLUMNS
            for horizon in HORIZONS
        ]
    )


scores = importance_scores(importance)
expected_score_rows = len(FEATURE_COLUMNS) * len(TARGET_COLUMNS) * OUTPUT_HORIZONS
if len(scores) != expected_score_rows:
    raise ValueError(f"Expected {expected_score_rows} score rows, got {len(scores)}")
scores

## Ranking against the placebo
Both scores carry their target's units, so features compare within a target, never across targets. Averaging each score over the horizons gives one number per feature and target, and sorting by the mean shift ranks the features for each target. The chart below draws that ranking with one panel per target, top feature at the top, and the placebo in gray.

The placebo is the yardstick. A model that reads all columns jointly can react to any scrambled column, so the placebo's bar is not zero; it shows how large that reaction is on the deployment. A feature counts as a driver only when its bar clearly exceeds the placebo's. With your own data, append a random-walk column as the placebo.

The two scores answer different questions. `driver` acts on the level of `target`, so the mean shift is the score that singles it out. `distance` centers both sample sets first, so it ignores a level shift by construction and reports only whether shuffling a feature changes the spread or shape of the forecast.

Two more limits hold for any ranking:

- Importance is measured against everything else the model still sees: the targets' own history and the other features stay in place. Two inputs that carry the same information can therefore both score low, because shuffling one leaves the other.
- A high score says the forecast depends on the feature's history, not that the feature causes the target.

In [ ]:
def rank_features(scores):
    """Average each score over the horizons and rank features per target."""
    return (
        scores.groupby(["target", "feature"])[["mean", "distance"]]
        .mean()
        .sort_values(["target", "mean"], ascending=[True, False])
    )


ranking = rank_features(scores)
ranking

In [ ]:
import matplotlib.pyplot as plt

SURFACE = "#fcfcfb"
BAR_COLOR = "#2a78d6"
PLACEBO_COLOR = "#898781"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
TEXT_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
BAR_HEIGHT = 0.4
PANEL_WIDTH_INCHES = 6.0
PANEL_HEIGHT_INCHES = 2.6

figure, axes = plt.subplots(
    1,
    len(TARGET_COLUMNS),
    figsize=(PANEL_WIDTH_INCHES * len(TARGET_COLUMNS), PANEL_HEIGHT_INCHES),
    squeeze=False,
    layout="constrained",
    facecolor=SURFACE,
)
for axis, target in zip(axes[0], TARGET_COLUMNS, strict=True):
    shifts = ranking.loc[target, "mean"].sort_values()
    colors = [
        PLACEBO_COLOR if feature == PLACEBO_COLUMN else BAR_COLOR
        for feature in shifts.index
    ]
    axis.barh(shifts.index, shifts.to_numpy(), height=BAR_HEIGHT, color=colors)
    axis.set_facecolor(SURFACE)
    axis.set_title(target, loc="left", color=TEXT_PRIMARY)
    axis.set_xlabel(
        "mean shift in target units, averaged over horizons", color=TEXT_MUTED
    )
    axis.ticklabel_format(axis="x", style="sci", scilimits=(-3, 4), useMathText=True)
    axis.xaxis.get_offset_text().set_color(TEXT_MUTED)
    axis.grid(axis="x", color=GRIDLINE, linewidth=1)
    axis.set_axisbelow(True)
    axis.tick_params(axis="x", colors=TEXT_MUTED, length=0)
    axis.tick_params(axis="y", labelcolor=TEXT_SECONDARY, length=0)
    for side in ("top", "right", "bottom"):
        axis.spines[side].set_visible(False)
    axis.spines["left"].set_color(BASELINE)
figure.suptitle("Forecast mean shift when a feature is shuffled", color=TEXT_PRIMARY)
plt.show()

## Checking that the ranking holds
Averaging over `n_repeats` shuffles evens out lucky and unlucky shuffles, but the scores still move from run to run. A rerun at another `seed` draws new shuffles and new samples: a gap between two features that keeps its sign across runs is real, and one that flips is a tie. Pick the rerun seed far from the first, because a request split into batches seeds batch `i` with `seed + i`, and seeds close together share most batches. A larger `n_repeats` steadies the averages; a larger `n_samples` shrinks only the sampling part of the noise.

In [ ]:
RERUN_SEED = 1_000_007

rerun = client.feature_importance(
    history,
    query_times=QUERY_TIMES,
    horizons=HORIZONS,
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    columns=plan.columns,
    n_samples=N_SAMPLES,
    n_repeats=N_REPEATS,
    seed=RERUN_SEED,
)
stability = pd.concat(
    {"first run": ranking, "rerun": rank_features(importance_scores(rerun))},
    axis=1,
)
stability